# Small Artificial Neural Network

In [1]:
import pandas as pd
import numpy as np
import joblib
import time

In [2]:
import sys
print(sys.executable)

C:\Users\Admin\.virtualenvs\ai_customer_intelligence-3eJa7RW-\Scripts\python.EXE


In [3]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.18.0
GPUs: []


In [4]:
X_train = joblib.load("../models/X_train_engineered_processed.pkl")
X_test = joblib.load("../models/X_test_engineered_processed.pkl")
y_train = joblib.load("../models/y_train.pkl")
y_test = joblib.load("../models/y_test.pkl")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 75)
X_test : (1409, 75)
y_train: (5634,)
y_test : (1409,)


In [5]:
y_train_ann = (y_train == "Yes").astype(int)
y_test_ann = (y_test == "Yes").astype(int)

In [6]:
print(y_train_ann.value_counts())
print(y_test_ann.value_counts())

Churn
0    4139
1    1495
Name: count, dtype: int64
Churn
0    1035
1     374
Name: count, dtype: int64


In [7]:
ann_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64,activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32,activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1,activation="sigmoid")
])

In [8]:
ann_model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

#### Calculate class weights

In [9]:
negative_count = (y_train_ann == 0).sum()
positive_count = (y_train_ann == 1).sum()
total = negative_count + positive_count
class_weight = {
    0: total / (2 * negative_count),
    1: total / (2 * positive_count)
}
print("Class weights:", class_weight)

Class weights: {0: np.float64(0.6805991785455424), 1: np.float64(1.8842809364548494)}


#### Early stopping

In [10]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [11]:
start_time = time.time()
history = ann_model.fit(
    X_train,
    y_train_ann,
    validation_split=0.20,
    epochs=30,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[early_stopping],
    verbose=1,
    shuffle=True
)
ann_training_time = time.time() - start_time

Epoch 1/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.7067 - loss: 0.5580 - val_accuracy: 0.7143 - val_loss: 0.5323
Epoch 2/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7420 - loss: 0.5098 - val_accuracy: 0.7178 - val_loss: 0.5141
Epoch 3/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7539 - loss: 0.4888 - val_accuracy: 0.7169 - val_loss: 0.5193
Epoch 4/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7499 - loss: 0.4976 - val_accuracy: 0.7311 - val_loss: 0.5023
Epoch 5/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7528 - loss: 0.4864 - val_accuracy: 0.7329 - val_loss: 0.4990
Epoch 6/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7650 - loss: 0.4812 - val_accuracy: 0.7267 - val_loss: 0.5125
Epoch 7/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7633 - loss: 0.4839 - val_accuracy: 0.7258 - val_loss: 0.5131
Epoch 8/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7666 - loss: 0.4743 - val_accuracy: 0.

In [12]:
start_time = time.time()
y_proba_ann = ann_model.predict(X_test,verbose=0).ravel()
ann_inference_time = time.time() - start_time

In [13]:
y_pred_ann = (y_proba_ann >= 0.5).astype(int)

In [14]:
y_pred_ann_labels = np.where(y_pred_ann == 1,"Yes","No")

#### Evaluate the ANN

In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [16]:
ann_accuracy = accuracy_score(y_test,y_pred_ann_labels)
ann_precision = precision_score(y_test,y_pred_ann_labels,pos_label="Yes",zero_division=0)
ann_recall = recall_score(y_test,y_pred_ann_labels,pos_label="Yes",zero_division=0)
ann_f1 = f1_score(y_test,y_pred_ann_labels,pos_label="Yes",zero_division=0)
ann_roc_auc = roc_auc_score(y_test_ann,y_proba_ann)

In [17]:
from sklearn.metrics import average_precision_score
ann_pr_auc = average_precision_score(y_test_ann,y_proba_ann)

In [18]:
ann_results = pd.DataFrame([
    {
        "Model": "Small ANN",
        "Accuracy": ann_accuracy,
        "Precision": ann_precision,
        "Recall": ann_recall,
        "F1-Score": ann_f1,
        "ROC-AUC": ann_roc_auc,
        "PR-AUC": ann_pr_auc,
        "Training Time (sec)": ann_training_time,
        "Inference Time (sec)": ann_inference_time
    }
])

ann_results = ann_results.round(4)
ann_results

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC,Training Time (sec),Inference Time (sec)
0,Small ANN,0.7637,0.5382,0.7727,0.6345,0.8383,0.6145,12.1524,0.4156


In [19]:
ann_results.to_csv("../models/phase7_ann_results.csv",index=False)
print("ANN results saved successfully.")

ANN results saved successfully.
